# Bio-Lab AI — Mistral 7B QLoRA
This notebook trains **only a rank-8 LoRA adapter** in Google Colab. It refuses production training with fewer than 200 scientist-approved examples or missing task coverage. Do not run it on the local 8 GB Mac.

In [ ]:
!nvidia-smi
!pip install -q 'transformers>=4.46,<5' 'trl>=0.12,<1' 'peft>=0.13,<1' 'bitsandbytes>=0.43' 'datasets>=3,<5' 'accelerate>=1,<2' 'huggingface_hub>=0.26,<1' 'trackio>=0.2'

In [ ]:
from google.colab import files
from huggingface_hub import notebook_login, whoami

BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
REQUIRED_TASKS = {
    'experiment_analysis', 'data_analysis', 'experiment_chat',
    'experiment_comparison', 'protocol_generation', 'sop_structuring',
    'project_chat', 'project_synthesis', 'general_chat',
}
notebook_login()  # use a write token; the adapter repository is private
HUB_MODEL_ID = f"{whoami()['name']}/biolab-ai-mistral-lora"
print(f'Adapter destination: {HUB_MODEL_ID} (private)')

In [ ]:
uploaded = files.upload()
dataset_path = next(name for name in uploaded if name.endswith('.jsonl'))

import json
rows = [json.loads(line) for line in uploaded[dataset_path].decode('utf-8').splitlines() if line.strip()]
assert len(rows) >= 200, f'Need at least 200 approved examples; found {len(rows)}.'
task_types = {row.get('task_type') for row in rows}
missing = REQUIRED_TASKS - task_types
assert not missing, f'Missing task coverage: {sorted(missing)}'
assert all(row.get('split') in {'train', 'validation', 'test'} for row in rows)
assert all(isinstance(row.get('messages'), list) and row['messages'][-1].get('role') == 'assistant' for row in rows)
print(f'Validated {len(rows)} approved examples across {len(task_types)} tasks.')

In [ ]:
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

assert torch.cuda.is_available(), 'A Colab GPU runtime is required. Do not run this on the Mac.'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = 'right'

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quantization, device_map='auto')
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

In [ ]:
def render(example):
    return {
        **example,
        'text': tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False),
    }

splits = {}
for split in ('train', 'validation', 'test'):
    split_rows = [row for row in rows if row['split'] == split]
    assert split_rows, f'The grouped export produced no {split} examples.'
    splits[split] = Dataset.from_list(split_rows).map(render)
dataset = DatasetDict(splits)
dataset

In [ ]:
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM', target_modules=['q_proj', 'v_proj'],
)
config = SFTConfig(
    output_dir='biolab-ai-mistral-lora',
    max_length=2048,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True, bf16=False,
    logging_steps=5,
    eval_strategy='steps', eval_steps=25,
    save_strategy='steps', save_steps=25, save_total_limit=2,
    report_to='trackio', project='biolab-ai', run_name='mistral-7b-r8-v1',
    push_to_hub=False,
    dataset_text_field='text', packing=False,
)
trainer = SFTTrainer(
    model=model, args=config, peft_config=lora, processing_class=tokenizer,
    train_dataset=dataset['train'], eval_dataset=dataset['validation'],
)
trainer.train()
print(trainer.evaluate(dataset['test']))

In [ ]:
import json, pathlib
from huggingface_hub import HfApi

adapter_dir = pathlib.Path('biolab-ai-mistral-lora-adapter')
trainer.model.save_pretrained(str(adapter_dir), safe_serialization=True)
config_path = adapter_dir / 'adapter_config.json'
adapter_config = json.loads(config_path.read_text())
adapter_config['model_type'] = 'mistral'  # required by Workers AI LoRA upload
config_path.write_text(json.dumps(adapter_config, indent=2) + '\n')
api = HfApi()
api.create_repo(repo_id=HUB_MODEL_ID, repo_type='model', private=True, exist_ok=True)
api.upload_folder(repo_id=HUB_MODEL_ID, repo_type='model', folder_path=str(adapter_dir))
print(f'Private adapter saved at https://huggingface.co/{HUB_MODEL_ID}')
print('Next: complete the human release gates, then upload adapter_config.json and adapter_model.safetensors to Cloudflare.')